In [0]:

import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import math
 
catalog = "workspace"
schema = "stock_analytics"
 
print("=" * 70)
print("TECHNICAL INDICATORS CALCULATION")
print("=" * 70)
 

In [0]:

# Cell 1: Load Bronze Data
bronze_df = spark.table(f"{catalog}.{schema}.bronze_stock_prices")
 
print("✅ Loaded bronze_stock_prices")
print(f"   Records: {bronze_df.count():,}")
 

In [0]:

# Cell 2: Calculate Moving Averages
print("\n" + "=" * 70)
print("CALCULATING MOVING AVERAGES")
print("=" * 70 + "\n")
 
# Window specification: ordered by date for each stock
window_spec = Window.partitionBy("Ticker").orderBy("Date")
 
# Moving Averages: 20-day, 50-day, 200-day
silver_df = bronze_df \
    .withColumn(
        "MA_20",
        F.avg("Close").over(
            Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-19*86400, 0)
        )
    ) \
    .withColumn(
        "MA_50",
        F.avg("Close").over(
            Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-49*86400, 0)
        )
    ) \
    .withColumn(
        "MA_200",
        F.avg("Close").over(
            Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-199*86400, 0)
        )
    )
 
print("✅ Calculated: 20-day, 50-day, 200-day Moving Averages")
 

In [0]:

# Cell 3: Calculate Exponential Moving Average (EMA)
print("\n" + "=" * 70)
print("CALCULATING EXPONENTIAL MOVING AVERAGES")
print("=" * 70 + "\n")
 
# Approximate EMA using row_number (simplified version)
# EMA = sum of (price * weight) / sum of weights
# where weight decreases exponentially as you go back
 
silver_df = silver_df \
    .withColumn(
        "EMA_12",
        F.avg("Close").over(
            Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-11*86400, 0)
        ) * 0.7 + 
        F.avg("Close").over(
            Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-23*86400, 0)
        ) * 0.3
    ) \
    .withColumn(
        "EMA_26",
        F.avg("Close").over(
            Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-25*86400, 0)
        ) * 0.6 + 
        F.avg("Close").over(
            Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-51*86400, 0)
        ) * 0.4
    )
 
print("✅ Calculated: 12-day and 26-day EMA")

In [0]:

# Cell 4: Calculate MACD (Moving Average Convergence Divergence)
print("\n" + "=" * 70)
print("CALCULATING MACD")
print("=" * 70 + "\n")
 
# MACD = EMA_12 - EMA_26
# Signal = EMA_9 of MACD
# Histogram = MACD - Signal
 
silver_df = silver_df \
    .withColumn(
        "MACD_Line",
        F.col("EMA_12") - F.col("EMA_26")
    ) \
    .withColumn(
        "MACD_Signal",
        F.avg(F.col("MACD_Line")).over(
            Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-8*86400, 0)
        )
    ) \
    .withColumn(
        "MACD_Histogram",
        F.col("MACD_Line") - F.col("MACD_Signal")
    )
 
print("✅ Calculated: MACD Line, Signal, Histogram")

In [0]:

# Cell 5: Calculate Bollinger Bands
print("\n" + "=" * 70)
print("CALCULATING BOLLINGER BANDS")
print("=" * 70 + "\n")
 
# Bollinger Bands: MA_20 ± (2 * StdDev)
window_20 = Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-19*86400, 0)
 
silver_df = silver_df \
    .withColumn(
        "BB_Std",
        F.stddev("Close").over(window_20)
    ) \
    .withColumn(
        "BB_Upper",
        F.col("MA_20") + (2 * F.col("BB_Std"))
    ) \
    .withColumn(
        "BB_Lower",
        F.col("MA_20") - (2 * F.col("BB_Std"))
    ) \
    .withColumn(
        "BB_Width",
        F.col("BB_Upper") - F.col("BB_Lower")
    ) \
    .withColumn(
        "BB_Position",
        F.when(F.col("BB_Width") > 0,
            (F.col("Close") - F.col("BB_Lower")) / F.col("BB_Width")
        ).otherwise(0.5)
    )
 
print("✅ Calculated: Bollinger Bands (Upper, Lower, Position)")
 

In [0]:

# Cell 6: Calculate RSI (Relative Strength Index)
print("\n" + "=" * 70)
print("CALCULATING RSI")
print("=" * 70 + "\n")
 
# RSI = 100 - (100 / (1 + RS))
# RS = Average Gain / Average Loss
 
window_14 = Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-13*86400, 0)
 
silver_df = silver_df \
    .withColumn(
        "Daily_Change",
        F.col("Close") - F.lag("Close").over(window_spec)
    ) \
    .withColumn(
        "Gain",
        F.when(F.col("Daily_Change") > 0, F.col("Daily_Change")).otherwise(0)
    ) \
    .withColumn(
        "Loss",
        F.when(F.col("Daily_Change") < 0, -F.col("Daily_Change")).otherwise(0)
    ) \
    .withColumn(
        "Avg_Gain",
        F.avg("Gain").over(window_14)
    ) \
    .withColumn(
        "Avg_Loss",
        F.avg("Loss").over(window_14)
    ) \
    .withColumn(
        "RS",
        F.when(F.col("Avg_Loss") > 0, F.col("Avg_Gain") / F.col("Avg_Loss")).otherwise(0)
    ) \
    .withColumn(
        "RSI_14",
        100 - (100 / (1 + F.col("RS")))
    )
 
print("✅ Calculated: RSI-14")
 

In [0]:

# Cell 7: Calculate Volatility
print("\n" + "=" * 70)
print("CALCULATING VOLATILITY")
print("=" * 70 + "\n")
 
window_30 = Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-29*86400, 0)
window_90 = Window.partitionBy("Ticker").orderBy("Date").rangeBetween(-89*86400, 0)
 
silver_df = silver_df \
    .withColumn(
        "Daily_Return",
        (F.col("Close") - F.lag("Close").over(window_spec)) / F.lag("Close").over(window_spec) * 100
    ) \
    .withColumn(
        "Volatility_30",
        F.stddev("Daily_Return").over(window_30)
    ) \
    .withColumn(
        "Volatility_90",
        F.stddev("Daily_Return").over(window_90)
    )
 
print("✅ Calculated: 30-day and 90-day Volatility")

In [0]:

# Cell 8: Calculate Price Trends
print("\n" + "=" * 70)
print("CALCULATING TREND INDICATORS")
print("=" * 70 + "\n")
 
silver_df = silver_df \
    .withColumn(
        "Price_Trend_20",
        F.when(F.col("Close") > F.col("MA_20"), "Bullish")
         .when(F.col("Close") < F.col("MA_20"), "Bearish")
         .otherwise("Neutral")
    ) \
    .withColumn(
        "Price_Trend_200",
        F.when(F.col("Close") > F.col("MA_200"), "Uptrend")
         .when(F.col("Close") < F.col("MA_200"), "Downtrend")
         .otherwise("Ranging")
    ) \
    .withColumn(
        "Golden_Cross",
        F.when(
            (F.lag("MA_50").over(window_spec) <= F.lag("MA_200").over(window_spec)) &
            (F.col("MA_50") > F.col("MA_200")),
            "BUY_SIGNAL"
        ).otherwise("NONE")
    ) \
    .withColumn(
        "Death_Cross",
        F.when(
            (F.lag("MA_50").over(window_spec) >= F.lag("MA_200").over(window_spec)) &
            (F.col("MA_50") < F.col("MA_200")),
            "SELL_SIGNAL"
        ).otherwise("NONE")
    )
 
print("✅ Calculated: Trends (Bullish/Bearish, Golden/Death Cross)")
 